<a href="https://colab.research.google.com/github/Prerana-Bijekar/DL/blob/main/Practical-5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

In [6]:
# Step 1: Load dataset
from google.colab import files
uploaded=files.upload()
data=pd.read_csv("process_data.csv")

# Pivot so that each row = sample, columns = genes, values = FPKM
pivot_df = data.pivot(index="Sample", columns="Gene", values="FPKM")

# Merge with tissue labels
labels = data[["Sample", "tissue"]].drop_duplicates().set_index("Sample")
pivot_df = pivot_df.merge(labels, left_index=True, right_index=True)

# Features (gene expressions)
X = pivot_df.drop("tissue", axis=1).values

# Labels (breast tumor / normal)
y = LabelEncoder().fit_transform(pivot_df["tissue"].values)  # 0 = normal, 1 = breast tumor

Saving process_data.csv to process_data (1).csv


In [7]:
# Step 2: Preprocessing
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Reshape for RNN: [samples, timesteps, features]
X_scaled = X_scaled.reshape((X_scaled.shape[0], X_scaled.shape[1], 1))

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [8]:
# Step 3: Build RNN Model
model = Sequential()
model.add(SimpleRNN(64, activation='tanh', input_shape=(X_scaled.shape[1], 1)))
model.add(Dropout(0.3))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))  # Binary classification

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [9]:
# Step 4: Train Model
history = model.fit(X_train, y_train, epochs=30, batch_size=16,
                    validation_data=(X_test, y_test), verbose=1)

Epoch 1/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 13s 4s/step - accuracy: 0.4861 - loss: nan - val_accuracy: 0.5455 - val_loss: nan
Epoch 2/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 10s 4s/step - accuracy: 0.6364 - loss: nan - val_accuracy: 0.5455 - val_loss: nan
Epoch 3/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 9s 3s/step - accuracy: 0.5583 - loss: nan - val_accuracy: 0.5455 - val_loss: nan
Epoch 4/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 11s 3s/step - accuracy: 0.5583 - loss: nan - val_accuracy: 0.5455 - val_loss: nan
Epoch 5/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 10s 4s/step - accuracy: 0.6833 - loss: nan - val_accuracy: 0.5455 - val_loss: nan
Epoch 6/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 9s 3s/step - accuracy: 0.6286 - loss: nan - val_accuracy: 0.5455 - val_loss: nan
Epoch 7/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 9s 3s/step - accuracy: 0.5739 - loss: nan - val_accuracy: 0.5455 - val_loss: nan
Epoch 8/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 10s 4s/step - accuracy: 0.5817 - loss: nan - val_accuracy: 0.5455 - val_loss: nan
Epoch 9/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 10s 4s/step - accuracy:

In [10]:
# Step 5: Evaluate Model
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {acc*100:.2f}%")

y_pred = (model.predict(X_test) > 0.5).astype("int32")
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Test Accuracy: 54.55%
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 808ms/step

Classification Report:
               precision    recall  f1-score   support

           0       0.55      1.00      0.71         6
           1       0.00      0.00      0.00         5

    accuracy                           0.55        11
   macro avg       0.27      0.50      0.35        11
weighted avg       0.30      0.55      0.39        11


Confusion Matrix:
 [[6 0]
 [5 0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
